In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import ast

In [ ]:
print(plt.style.available)

In [ ]:
log_path = "/home/yanhongwei/DGIL/DGIL/logs/l2p/officecaltech/0/2/CIL_1993_vit_base_patch16_224_l2p.log"

In [ ]:
Dataset_DomainName_Mapping = {
    "imageclef": ["i", "p", "c"],
    "domainnet": ["clipart", "infograph", "painting", "quickdraw", "real", "sketch"],
    "minidomainnet": ["clipart", "painting", "real", "sketch"],
    "officehome": ["Art", "Clipart", "Product", "Real World"],
    "office31": ["amazon", "dslr", "webcam"],
    "officecaltech": ["amazon", "caltech", "dslr", "webcam"],
    "digitsdg": ["mnist", "mnist_m", "svhn", "syn"],
    "digitsfive": ["mnist", "mnist_m", "svhn", "syn", "usps"],
    "core50": [f"s{i}" for i in range(1, 12)]
}

In [ ]:
average_cnn_prefix = "[trainer.py] => CNN: "
average_nme_prefix = "[trainer.py] => NME: "

for key in Dataset_DomainName_Mapping.keys():
    if f'/{key}/' in log_path:

        domain_names = Dataset_DomainName_Mapping[key]
        domain_cnn_prefix_map = {}
        domain_nme_prefix_map = {}

        for domain_id, domain_name in enumerate(domain_names):
            domain_cnn_prefix_map[f"{domain_name}_cnn"] = f"[trainer.py] => Domain [{domain_id}] {domain_name}: CNN: "
            domain_nme_prefix_map[f"{domain_name}_nme"] = f"[trainer.py] => Domain [{domain_id}] {domain_name}: NME: "

        break

all_prefix_map = {
    "average_cnn": average_cnn_prefix,
    "average_nme": average_nme_prefix,
    **domain_cnn_prefix_map,
    **domain_nme_prefix_map
}

plt.style.use('seaborn-v0_8-talk')

In [ ]:
def load_data(file_path, prefix_str):
    with open(file_path, 'r') as f:
        lines = f.readlines()
    acc_dict_ls = []
    for line in lines:
        if prefix_str in line:
            accy_dict_str = line.split(prefix_str)[1]
            accy_dict = ast.literal_eval(accy_dict_str)
            acc_dict_ls.append(accy_dict)
    return acc_dict_ls

def split_data(acc_dict_ls):
    acc_ls_dict = {}
    for acc_dict in acc_dict_ls:
        for key, value in acc_dict.items():
            if key not in acc_ls_dict:
                acc_ls_dict[key] = []
            acc_ls_dict[key].append(value)
    return acc_ls_dict

def load_domain_data(file_path, prefix_str_ls):
    domain_acc_dict_ls = {domain_name: [] for domain_name in domain_names}
    with open(file_path, 'r') as f:
        lines = f.readlines()
    for line in lines:
        for prefix_str in prefix_str_ls:
            if prefix_str in line:
                for domain_id, domain_name in enumerate(domain_names):
                    if f"[trainer.py] => Domain [{domain_id}] {domain_name}:" in line:
                        accy_dict_str = line.split(prefix_str)[1]
                        accy_dict = ast.literal_eval(accy_dict_str)
                        domain_acc_dict_ls[domain_name].append(accy_dict)
    return domain_acc_dict_ls

def split_domain_data(domain_acc_dict_ls):
    domain_acc_ls_dict = {domain_name: {} for domain_name in domain_names}
    for domain_name, acc_dict_ls in domain_acc_dict_ls.items():
        for acc_dict in acc_dict_ls:
            for key, value in acc_dict.items():
                if key not in domain_acc_ls_dict[domain_name]:
                    domain_acc_ls_dict[domain_name][key] = []
                domain_acc_ls_dict[domain_name][key].append(value)
    return domain_acc_ls_dict

In [ ]:
def plot_global_acc(acc_ls_dict, title = 'Global Accuracy'):
    plt.figure(figsize=(10, 5))

    total_acc_ls = acc_ls_dict['total']
    old_acc_ls = acc_ls_dict['old']
    new_acc_ls = acc_ls_dict['new']
    plt.plot(total_acc_ls, label='Total Accuracy', linewidth=2, marker='o')
    plt.plot(old_acc_ls, label='Old Accuracy', linewidth=2, marker='o')
    plt.plot(new_acc_ls, label='New Accuracy', linewidth=2, marker='o')

    plt.xlabel('Task', fontdict={'size':15})
    plt.ylabel('Accuracy(%)', fontdict={'size':15})
    plt.xticks(fontsize=14)
    plt.yticks(fontsize=14)
    plt.ylim(0, 100)
    plt.legend(fontsize=14)
    plt.title(title, fontdict={'size':15})
    plt.show()

def plot_task_acc(acc_ls_dict, title = 'Task-wise Accuracy'):
    plt.figure(figsize=(10, 5))

    task_num = len(acc_ls_dict['total'])
    for task_id, acc_ls in acc_ls_dict.items():
        if '-' in task_id:
            acc_ls_task = acc_ls_dict[task_id]
            start_task = task_num - len(acc_ls_task)
            plt.plot(range(start_task, start_task+len(acc_ls_task)), acc_ls_task, label=task_id, linewidth=2, marker='o')

    plt.xlabel('Task', fontdict={'size':15})
    plt.ylabel('Accuracy(%)', fontdict={'size':15})
    plt.xticks(fontsize=14)
    plt.yticks(fontsize=14)
    plt.ylim(0, 100)
    plt.legend(fontsize=14)
    plt.title(title, fontdict={'size':15})
    plt.show()

def plot_domain_acc(domain_acc_ls_dict, key='total'):

    plt.figure(figsize=(10, 5))

    for domain in domain_names:
        acc_ls = domain_acc_ls_dict[domain][key]
        plt.plot(acc_ls, label=domain, linewidth=2, marker='o')

    plt.xlabel('Task', fontdict={'size':15})
    plt.ylabel('Accuracy(%)', fontdict={'size':15})
    plt.xticks(fontsize=14)
    plt.yticks(fontsize=14)
    plt.ylim(0, 100)
    plt.legend(fontsize=14)
    plt.title(f'Domain-wise Accuracy ({key})', fontdict={'size':15})
    plt.show()

In [ ]:
def plot_all_acc_data(log_path, all_prefix_map, cls_head="cnn"):
    for prefix_name, prefix in all_prefix_map.items():
        if f"_{cls_head}" in prefix_name:
            acc_dict_ls = load_data(log_path, prefix)
            acc_ls_dict = split_data(acc_dict_ls)
            plot_global_acc(acc_ls_dict, f"Global Acc of {prefix_name}")
            # plot_task_acc(acc_ls_dict, f"Task Acc of {prefix_name}")

def plot_domain_wise_acc_data(log_path, all_prefix_map, cls_head="cnn"):
    prefix_str_ls = []
    for prefix_name, prefix in all_prefix_map.items():
        for domain_name in domain_names:
            if f"{domain_name}_{cls_head}" in prefix_name:
                prefix_str_ls.append(prefix)
    domain_acc_dict_ls = load_domain_data(log_path, prefix_str_ls)
    domain_acc_ls_dict = split_domain_data(domain_acc_dict_ls)
    plot_domain_acc(domain_acc_ls_dict, key='total')
    plot_domain_acc(domain_acc_ls_dict, key='old')
    plot_domain_acc(domain_acc_ls_dict, key='new')

In [ ]:
# plot_all_acc_data(log_path, all_prefix_map, cls_head="cnn")
plot_domain_wise_acc_data(log_path, all_prefix_map, cls_head="cnn")

In [ ]:
log_path = "/home/yanhongwei/DGIL/DGIL/logs/l2p/officecaltech/0/2/DGIL_1993_vit_base_patch16_224_l2p.log"
plot_domain_wise_acc_data(log_path, all_prefix_map, cls_head="cnn")

In [ ]:
log_path = "/home/yanhongwei/DGIL/DGIL/logs/l2p/officecaltech/0/2/DGIL_v2_1993_vit_base_patch16_224_l2p.log"
plot_domain_wise_acc_data(log_path, all_prefix_map, cls_head="cnn")

In [1]:
import os
import pandas as pd

def parse_log_file(log_file, eval_key='CNN'):
    is_valid = True
    result_dict = {
        "model_name": "",
        "dataset": "",
        "prefix": "",
        "seed": "",
        "average_accuracy_all": 0.0,
        "last_accuracy_all": 0.0,
        "last_accuracy_in": 0.0,
        "last_accuracy_out": 0.0,
        "last_accuracy_out_worst": 0.0,
        "forgetting": -1e-5,
    }
    
    with open(log_file, 'r') as f:
        lines = f.readlines()
    
    task_reference_domains = {}
    class_id_pairs = []
    domain_accuracies = {}
    no_nme = False
    
    for line in lines:
        if "No NME" in line and eval_key == "NME":
            no_nme = True
        if "[trainer.py] => model_name:" in line:
            result_dict["model_name"] = line.split(":")[-1].strip()
        if "[trainer.py] => prefix" in line:
            result_dict["prefix"] = line.split(":")[-1].strip()
        if "[trainer.py] => seed:" in line:
            result_dict["seed"] = line.split(":")[-1].strip()
        if "[trainer.py] => dataset:" in line:
            result_dict["dataset"] = line.split(":")[-1].strip()
        if f"[trainer.py] => Average Accuracy ({eval_key}):" in line:
            result_dict["average_accuracy_all"] = float(line.split(":")[-1].strip())
        if f"[trainer.py] => Last Accuracy ({eval_key}):" in line:
            result_dict["last_accuracy_all"] = float(line.split(":")[-1].strip())
        if (f"[trainer.py] => {eval_key}"+": {'total':" )in line:
            cnn_dict = eval(line.split(f"{eval_key}:")[-1].strip())
            result_dict["last_accuracy_all"] = cnn_dict.get("total", 0.0)
        if f"[trainer.py] => Forgetting ({eval_key}):" in line:
            result_dict["forgetting"] = float(line.split(":")[-1].strip())
        if "=> Task" in line and "reference domain is" in line:
            task_id = int(line.split("Task")[1].split(":")[0].strip())
            domain_part = line.split("reference domain is")[-1].strip()
            domain_id = int(domain_part.split("[")[1].split("]")[0].strip())
            task_reference_domains[task_id] = domain_id
        if "[trainer.py] => Class ID pairs:" in line:
            class_id_pairs = eval(line.split(":")[-1].strip())
        if "[trainer.py] => Domain [" in line and f"{eval_key}:" in line:
            domain_part = line.split("[trainer.py] => Domain")[-1].strip()
            domain_id = int(domain_part.split("[")[1].split("]")[0].strip())
            accuracies = eval(line.split(f"{eval_key}:")[-1].strip())
            domain_accuracies[domain_id] = accuracies

    # assert result_dict["forgetting"] > 0.0, f"invalid log file: {log_file}"
    if no_nme:
        print(f"No NME for {log_file}")
        is_valid = False
    elif result_dict["forgetting"] < 0.0:
        print(f"incomplete log file: {log_file}")
        is_valid = False
    
    if not is_valid:
        return result_dict, False

    # Calculate last_accuracy_in, last_accuracy_out, and last_accuracy_out_worst
    in_accuracies = []
    out_accuracies = []
    out_worst_accuracies = []
    
    for task_id, ref_domain in task_reference_domains.items():
        class_pair = class_id_pairs[task_id]
        class_key = f"{class_pair[0]:02d}-{class_pair[1]:02d}"
        
        # In-domain accuracy
        in_acc = domain_accuracies[ref_domain].get(class_key, 0.0)
        in_accuracies.append(in_acc)
        
        # Out-domain accuracy
        out_domains = [domain for domain in domain_accuracies.keys() if domain != ref_domain]
        out_accs = [domain_accuracies[domain].get(class_key, 0.0) for domain in out_domains]
        out_accuracies.append(sum(out_accs) / len(out_accs))
        
        # Worst out-domain accuracy
        out_worst_accuracies.append(min(out_accs))
    
    result_dict["last_accuracy_in"] = sum(in_accuracies) / len(in_accuracies) if in_accuracies else 0.0
    result_dict["last_accuracy_out"] = sum(out_accuracies) / len(out_accuracies) if out_accuracies else 0.0
    result_dict["last_accuracy_out_worst"] = sum(out_worst_accuracies) / len(out_worst_accuracies) if out_worst_accuracies else 0.0
    
    return result_dict, is_valid


def parse_log_folder(log_folder, eval_key='CNN'):

    all_result = []

    for root, dirs, files in os.walk(log_folder):
        for file in files:
            if file.endswith(".log"):
                log_file = os.path.join(root, file)
                result_dict, is_valid = parse_log_file(log_file, eval_key)
                if is_valid: all_result.append(result_dict)

    # Rearrange the contents of the csv file in the following form:
    # method | prefix | seed | dataset | average_accuracy_all | last_accuracy_all | last_accuracy_in | last_accuracy_out | last_accuracy_out_worst | forgetting
    df = pd.DataFrame(all_result)
    df = df[["model_name", "prefix", "seed", "dataset", "average_accuracy_all", "last_accuracy_all", "last_accuracy_in", "last_accuracy_out", "last_accuracy_out_worst", "forgetting"]]
    df = df.sort_values(by=["model_name", "dataset", "prefix", "seed"])
    df.to_csv(f"{eval_key}_results.csv", index=False)


    # Filter seeds to include only [1994, 1995, 1996]
    # valid_seeds = ["1993"]
    valid_seeds = ["1994", "1995", "1996"]
    df_filtered = df[df["seed"].astype(str).isin(valid_seeds)]

    # valid_model_name = ["dot"]
    # df_filtered = df_filtered[df_filtered["model_name"].isin(valid_model_name)]

    # Save the filtered results to a new CSV file
    df_filtered.to_csv(f"{eval_key}_results_filtered.csv", index=False)

    # Calculate mean and std for each setup (grouped by model_name, prefix, dataset) for the filtered seeds
    mean_std_df = df_filtered.groupby(["model_name", "prefix", "dataset"]).agg({
        "average_accuracy_all": ["mean", "std"],
        "last_accuracy_all": ["mean", "std"],
        "last_accuracy_in": ["mean", "std"],
        "last_accuracy_out": ["mean", "std"],
        "last_accuracy_out_worst": ["mean", "std"],
        "forgetting": ["mean", "std"],
    }).reset_index()


    # Flatten the multi-level columns
    mean_std_df.columns = ["_".join(col).strip() for col in mean_std_df.columns.values]

    # Save the mean and std results to a new CSV file
    mean_std_df.to_csv(f"{eval_key}_results_mean_std_filtered.csv", index=False)


root_folder = "/gpfs-flash/junlab/liyuan/hongwei/DGIL"
parse_log_folder(root_folder, eval_key='CNN')
# parse_log_folder(root_folder, eval_key='NME')

incomplete log file: /gpfs-flash/junlab/liyuan/hongwei/DGIL/logs/l2p/core50/0/5/CIL_p30_1996_vit_base_patch16_224_l2p.log
incomplete log file: /gpfs-flash/junlab/liyuan/hongwei/DGIL/logs/l2p/domainnet/0/35/DGIL_p30_1994_vit_base_patch16_224_l2p.log
incomplete log file: /gpfs-flash/junlab/liyuan/hongwei/DGIL/logs/l2p/domainnet/0/35/CIL_p30_1994_vit_base_patch16_224_l2p.log
incomplete log file: /gpfs-flash/junlab/liyuan/hongwei/DGIL/logs/dot_slca/domainnet/0/35/DGIL_dot_e5_supcon0.1_dom10_1994_vit_base_patch16_224_dot.log
incomplete log file: /gpfs-flash/junlab/liyuan/hongwei/DGIL/logs/dot_slca/domainnet/0/35/DGIL_dot_e2_supcon0.1_dom10_1995_vit_base_patch16_224_dot.log
incomplete log file: /gpfs-flash/junlab/liyuan/hongwei/DGIL/logs/dot_slca/domainnet/0/35/DGIL_dot_e2_supcon0.1_dom0.1_1996_vit_base_patch16_224_dot.log
incomplete log file: /gpfs-flash/junlab/liyuan/hongwei/DGIL/logs/dot_slca/domainnet/0/35/DGIL_dot_supcon0.1_dom0.1_1996_vit_base_patch16_224_dot.log
incomplete log file: /